# Exploring Spatial Clustering Stability using Cross Method Entropy 

## 1. Introduction 

### 1.1 Background 

Spatial transcriptomics is a technology that measures gene expression in individual cells while preserving their spatial location in the tissue. In this project, we work with MERFISH data from the mouse hypothalamus preoptic region.
A common analysis step is spatial clustering, grouping cells with similar gene expression and spatial patterns into distinct regions. However, spatial clustering is an exploratory task with no established guidelines, and results depend heavily on parameter choices, making it difficult to distinguish true biology from parameter artifacts.
We use SpatialLeiden, an extension of the Leiden clustering algorithm that incorporates spatial information through additional parameters controlling the balance between gene expression and spatial location.
To assess the stability of clustering results across different parameter settings, we use Cross Method Entropy (CME), introduced by Sun et al. 2026. CME produces a map where each cell is assigned an entropy value reflecting how consistently it is assigned to the same cluster across multiple runs.

### 1.2 Dataset

This project uses the MERFISH mouse hypothalamus 
preoptic region dataset, first published by Moffitt 
et al. 2018. The dataset contains:

- **5,488 cells** measured in the mouse hypothalamus
- **155 genes** measured per cell
- **Spatial coordinates** for each cell in the tissue


### 1.3 Objectives 

The objectives of this project are:

1. Reproduce the SpatialLeiden tutorial and understand 
   how the parameters affect clustering results.

2. Perform a systematic parameter scan by running 
   SpatialLeiden with combinations of layer ratio, 
   latent resolution, spatial resolution, and random 
   seed.

3. Apply the CME pipeline to produce stability maps 
   for the clustering results.

4. Run the original R CME implementation and a Python implementation and investigate the results.

5. Validate the CME maps against known biological 
   structures in the mouse hypothalamus.

## 2. SpatialLeiden vs. Leiden - Tutorial Reproduction

### 2.1 Objective 

To understand the difference between the standard Leiden clustering algorithm and SpatialLeiden, and to reproduce the SpatialLeiden tutorial on the MERFISH hypothalamus dataset.

### 2.2 Background 

The Leiden algorithm is a clustering method that groups cells based on the similarity of their gene expression profiles. It works by building a nearest neighbor graph where cells with similar gene expression are connected, and then finding communities within this graph.
SpatialLeiden extends Leiden by adding spatial information.


### 2.3 Method 

The MERFISH hypothalamus dataset was loaded and preprocessed. Both Leiden and SpatialLeiden were run on the same dataset with comparable parameters, and the resulting clusterings were visualized spatially to compare how each algorithm groups cells.


### 2.4 Code

In [ ]:
import scanpy as sc 
import spatialleiden as sl 
import squidpy as sq
import matplotlib.pyplot as plt
import os

# Create a folder for all plots 

os.makedirs("figures", exist_ok=True)

random_state = 42

adata = sc.read_h5ad("MERFISH_0.04.h5ad")
print(adata)

# Log transforming and PCA 

sc.pp.log1p(adata)
sc.pp.pca(adata, random_state = random_state)
sc.pp.neighbors(adata, random_state = random_state)

# Building spatial neighbor graphs

sq.gr.spatial_neighbors(adata, coord_type = "generic", n_neighs = 10)
adata.obsp["spatial_connectivities"] = sl.distance2connectivity(adata.obsp["spatial_distances"])

# Running Leiden and SpatialLeiden

sc.tl.leiden(adata, directed = False, random_state = random_state)

sl.spatialleiden(adata, layer_ratio = 1.8, directed = (False, True), random_state = random_state)

# Plotting and comparing 

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sc.pl.embedding(adata, basis="spatial", color="leiden",
                title="Leiden", ax=axes[0], show=False)

sc.pl.embedding(adata, basis="spatial", color="spatialleiden",
                title="SpatialLeiden", ax=axes[1], show=False)

plt.tight_layout()
plt.savefig("figures/02_leiden_vs_spatialleiden.png",
            dpi=150, bbox_inches="tight", facecolor="white")
plt.close()
print("Saved: figures/02_leiden_vs_spatialleiden.png")

### 2.5 Results

![Leiden vs SpatialLeiden](figures/02_leiden_vs_spatialleiden.png)

The figure above shows the clustering results of both algorithms plotted on the spatial coordinates of the tissue. 

Standard Leiden produced clusters based purely on gene expression similarity, without any consideration of where cells are located in the tissue. As a result, cells with similar gene expression but different spatial locations may be placed in the same cluster.

SpatialLeiden produced clusters that respect both gene expression similarity and spatial location, resulting in more spatially coherent regions. Known anatomical structures are more clearly separated as distinct clusters.

## 3. Parameter Scan and Biological Validation

### 3.1 Objective 

To investigate how the three main SpatialLeiden parameters (layer ratio, latent resolution, and spatial resolution), affect the clustering results, and to compare the resulting clusters to known biological structures in the mouse hypothalamus preoptic region.

### 3.2 Method 

Three separate parameter scans were performed, each varying one parameter while keeping the others fixed at a baseline value:

- **Layer ratio scan:** [1.4, 1.5, 1.6, 1.7, 1.8] with fixed latent resolution = 0.115 and spatial resolution = 0.9

- **Latent resolution scan:** [0.115, 0.138, 0.161, 0.184, 0.207] with fixed layer ratio = 1.4 and spatial resolution = 0.9

- **Spatial resolution scan:** [0.9, 1.05, 1.2, 1.35, 1.5] with fixed layer ratio = 1.4 and latent resolution = 0.115

For each parameter combination, the clustering result was visualized spatially and compared to known anatomical structures.

### 3.3 Expectation 

In all cases, we expect that at least some parameter combinations will produce clusters that correspond to known anatomical structures.

### 3.4 Code 

In [ ]:
# Layer ratio scanning
import os

n_clusters = adata.obs["domain"].nunique()
layer_ratios = [0.5, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5, 1.6, 1.7, 1.8, 1.9, 2.0, 2.5]

for ratio in layer_ratios:
    latent_res, spatial_res = sl.search_resolution(
        adata,
        n_clusters,
        latent_kwargs={"random_state": random_state},
        spatial_kwargs={
            "layer_ratio":  ratio,
            "random_state": random_state,
            "directed":     (False, True),
        },
    )
    key = f"leiden_ratio{ratio}"
    sl.spatialleiden(
        adata,
        resolution=(latent_res, spatial_res),
        layer_ratio=ratio,
        directed=(False, True),
        random_state=random_state,
        key_added=key,
    )
    print(f"ratio={ratio} -> latent={latent_res:.3f}, "
          f"spatial={spatial_res:.3f}")
# plot

keys  = [f"leiden_ratio{ratio}" for ratio in layer_ratios]
n     = len(keys)
ncols = 4
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols,
                          figsize=(ncols * 5, nrows * 5))
axes = axes.flatten()

for i, (key, ratio) in enumerate(zip(keys, layer_ratios)):
    k = adata.obs[key].nunique()
    sc.pl.embedding(
        adata,
        basis="spatial",
        color=key,
        title=f"layer_ratio={ratio}\nK={k}",
        ax=axes[i],        
        show=False,        
        legend_loc=None
    )

# hide empty subplots
for j in range(len(keys), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Layer Ratio Scan", fontsize=14)
plt.tight_layout()
plt.savefig("figures/03_layer_ratio_scan.png",  
            dpi=150, bbox_inches="tight",
            facecolor="white")
plt.close()
print("Saved: figures/03_layer_ratio_scan.png")

![Layer Ratio Scan](figures/03_layer_ratio_scan.png)

![layer_ratio atlas](figures/03_layer_ratio_atlas.png)

In [ ]:
# Latent resolution scanning 

import numpy as np
import matplotlib.pyplot as plt
import scanpy as sc

base_latent_res   = 0.138
latent_fractions  = np.linspace(0.5, 1.0, 11)
latent_resolutions = base_latent_res * latent_fractions

print("Latent resolutions to test:")
for f, r in zip(latent_fractions, latent_resolutions):
    print(f"  {f*100:.0f}% -> {r:.4f}")

spatial_res = 0.9
keys        = []

for latent_res in latent_resolutions:
    key = f"leiden_latent{latent_res:.4f}"
    keys.append(key)

    sl.spatialleiden(
        adata,
        resolution=(latent_res, spatial_res),
        layer_ratio=1.8,
        directed=(False, True),
        random_state=random_state,
        key_added=key,
    )

    n = adata.obs[key].nunique()
    print(f"latent_res={latent_res:.4f} "
          f"({latent_fractions[keys.index(key)]*100:.0f}%) "
          f"-> {n} clusters")

# plot
n     = len(keys)
ncols = 4
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols,
                          figsize=(ncols * 5, nrows * 5))
axes = axes.flatten()

for i, key in enumerate(keys):
    n_clust = adata.obs[key].nunique()
    sc.pl.embedding(
        adata,
        basis="spatial",
        color=key,
        title=f"latent={latent_resolutions[i]:.4f}\n"
              f"K={n_clust}",
        ax=axes[i],
        show=False,
        legend_loc=None
    )

for j in range(len(keys), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Latent Resolution Scan", fontsize=14)
plt.tight_layout()
plt.savefig("figures/03_latent_resolution_scan.png",
            dpi=150, bbox_inches="tight", facecolor="white")
plt.close()
print("Saved: figures/03_latent_resolution_scan.png")

![Latent Resolution Scan](figures/03_latent_resolution_scan.png)

![latent resolution atlas](figures/03_latent_resolution_atlas.png)

In [ ]:
# Spatial resolution scanning 

optimal_spatial    = 0.9
lower              = round(optimal_spatial * 0.5, 3)
upper              = round(optimal_spatial * 2.0, 3)
spatial_resolutions = [round(x, 3)
                        for x in np.linspace(lower, upper, 10)]
fixed_latent       = 0.138

print(f"Optimal spatial: {optimal_spatial}")
print(f"Range: {lower} to {upper}")
print(f"10 values: {spatial_resolutions}")

for spat_res in spatial_resolutions:
    key = f"sl_spat_{spat_res}"

    if key in adata.obs.columns:
        print(f"spatial={spat_res} already exists, skipping")
        continue

    sl.spatialleiden(
        adata,
        resolution=(fixed_latent, spat_res),
        layer_ratio=1.8,
        directed=(False, True),
        random_state=random_state,
        key_added=key
    )

    n = adata.obs[key].nunique()
    print(f"spatial={spat_res} → {n} clusters")

# plot
keys  = [f"sl_spat_{spat_res}"
         for spat_res in spatial_resolutions]
n     = len(keys)
ncols = 4
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols,
                          figsize=(ncols * 5, nrows * 5))
axes = axes.flatten()

for i, (key, spat_res) in enumerate(
        zip(keys, spatial_resolutions)):
    n_clust = adata.obs[key].nunique()
    sc.pl.embedding(
        adata,
        basis="spatial",
        color=key,
        title=f"spatial={spat_res}\nK={n_clust}",
        ax=axes[i],
        show=False,
        legend_loc=None
    )

for j in range(len(keys), len(axes)):
    axes[j].set_visible(False)

plt.suptitle(
    f"Spatial Resolution Scan ({lower} to {upper})",
    fontsize=14
)
plt.tight_layout()
plt.savefig("figures/03_spatial_resolution_scan.png",
            dpi=150, bbox_inches="tight", facecolor="white")
plt.close()
print("Saved: figures/03_spatial_resolution_scan.png")

![Spatial Resolution Scan](figures/03_spatial_resolution_scan.png)

![spatial resolution atlas](figures/03_spatial_resolution_atlas.png)

### 3.6 Interpretation 

The parameter scan confirms that all three parameters significantly affect the clustering results, which shows that spatial clustering of this dataset is highly parameter-sensitive.

Overall, the results are consistent with the expectation: higher resolutions produce more clusters and higher layer ratios produce more spatially coherent clusters. Known structures such as V3 are visible in the clustering results at appropriate parameter settings, suggesting that the algorithm is capturing real biological structure.

## 4. Parameter Permutation - Nested For Loop 

### 4.1 Objective 

To run SpatialLeiden with all combinations of four parameters (layer ratio, latent resolution, spatial resolution, and random seed) producing 625 clustering runs in total to be used as input for the CME analysis.

### 4.2 Method 

A nested for loop iterates over all combinations of the four parameters. For each combination, SpatialLeiden is run and the clustering result is stored. If a run already exists from a previous session, it is skipped to save computation time. After all runs are complete, the distribution of cluster counts is reported and the results are saved.


### 4.3 Expectation 

We expect:

- Different parameter combinations to produce different numbers of clusters.

- Higher resolution values to generally produce more clusters.

- Runs with the same parameters but different seeds to produce similar but not identical results.

- Most runs to produce between 6 and 10 clusters, which is a reasonable number for this dataset based on the known structures in the mouse hypothalamus.

### 4.4 Code

In [ ]:
# Nested for loop with optimal parameters

import numpy as np 
import matplotlib.pyplot as plt
import scanpy as sc 
import spatialleiden as sl
from collections import Counter

opt_layer_ratios = [1.4, 1.5, 1.6, 1.7, 1.8]
opt_latent_resolutions = [0.115, 0.138, 0.161, 0.184, 0.207]
opt_spatial_resolutions = [0.9, 1.05, 1.2, 1.35, 1.5]
random_seeds = [0, 42, 123, 456, 789]

total = 625
computed = 0
skipped = 0

for seed in random_seeds:
    for lay_rat in opt_layer_ratios:
        for lat_res in opt_latent_resolutions:
            for spat_res in opt_spatial_resolutions:

                key = f"sl_seed{seed}_ratio{lay_rat}_lat{lat_res}_spat{spat_res}"
                
                if key in adata.obs.columns:
                    skipped += 1
                    continue
                    
                sl.spatialleiden(
                    adata, 
                    resolution=(lat_res, spat_res),
                    layer_ratio=lay_rat,
                    directed=(False, True),
                    random_state=seed,
                    key_added=key
                )
                
                n = adata.obs[key].nunique()
                computed += 1
                
                # print progress every 50 runs
                if computed % 50 == 0:
                    print(f"Progress: {computed + skipped}/{total} "
                          f"({computed} computed, {skipped} skipped)")

print(f"\nDone! {computed} computed, {skipped} skipped")

# print cluster count distribution
sl_cols = [c for c in adata.obs.columns if c.startswith("sl_seed")]
cluster_counts = Counter([adata.obs[c].nunique() for c in sl_cols])
print(f"\nTotal runs: {len(sl_cols)}")
print("Runs per cluster count:")
for k, count in sorted(cluster_counts.items()):
    print(f"  K={k}: {count} runs")




### 4.5 Results 

All 625 parameter combinations were run successfully. The cluster counts ranged from K=4 to K=13, with most runs producing between 7 and 11 clusters. Only a small number of runs produced very few (K=4, 2 runs) or many clusters (K=13, 11 runs).

## 5. CME Pipeline 

### 5.1 Objective

To apply the Cross Method Entropy (CME) pipeline to the 625 clustering runs generated in the previous section and to compare the results between the original R script and the Python implementation.

### 5.2 Method

All 625 clustering runs were saved into a single input file, where each row is a cell and each column is one clustering run.

To select which runs to use for CME, 8 runs per cluster count were chosen by alternating between different random seeds to ensure diversity. Both the R and Python CME scripts were then run for each cluster count from K=4 to K=13 using these selected runs. K=4 was run separately since only 2 runs were available for this cluster count.

### 5.3 Expectation

Since both the R and Python scripts implement the same CME algorithm with the same input runs, we expect them to produce identical CME maps. 

### 5.4 Code 

In [ ]:
import pandas as pd
import subprocess
import os
import matplotlib.pyplot as plt
import scanpy as sc

os.makedirs("figures", exist_ok=True)

# aggregate all runs into one TSV

sl_cols = [c for c in adata.obs.columns
           if c.startswith("sl_seed")]
print(f"Total runs: {len(sl_cols)}")

label_df = adata.obs[sl_cols].copy()
label_df.to_csv("cme_input_all.tsv", sep="\t")
print("Saved: cme_input_all.tsv")

# create BC_rankings.tsv

df = pd.read_csv("cme_input_all.tsv", sep="\t", index_col=0)

cols_by_n = {}
for col in df.columns:
    n = df[col].nunique()
    if n not in cols_by_n:
        cols_by_n[n] = []
    cols_by_n[n].append(col)

bc_rankings = {}
for n, cols in sorted(cols_by_n.items()):
    bc_rankings[str(n)] = cols[:8]
    print(f"K={n}: {len(cols[:8])} runs selected")

max_len = max(len(v) for v in bc_rankings.values())
bc_df = pd.DataFrame({
    k: v + [None]*(max_len - len(v))
    for k, v in bc_rankings.items()
})
bc_df.to_csv("BC_rankings.tsv", sep="\t")
print("\nSaved: BC_rankings.tsv")

# run original R script

result_k4 = subprocess.run([
    "Rscript", "cme_script_original.R",
    "-i", "cme_input_all.tsv",
    "-o", "figures/cme_r_k4.tsv",
    "--n_bcs", "2",
    "-n", "4"
], capture_output=True, text=True)

print(f"R K=4: return code={result_k4.returncode}")
if result_k4.returncode != 0:
    print(f"ERROR: {result_k4.stderr[:200]}")

for k in [5, 6, 7, 8, 9, 10, 11, 12, 13]:
    result = subprocess.run([
        "Rscript", "cme_script_original.R",
        "-i", "cme_input_all.tsv",
        "-o", f"figures/cme_r_k{k}.tsv",
        "--BC_ranking", "BC_rankings.tsv",
        "--n_bcs", "8",
        "-n", str(k)
    ], capture_output=True, text=True)

    print(f"R K={k}: return code={result.returncode}")
    if result.returncode != 0:
        print(f"ERROR: {result.stderr[:200]}")

# load R results and add to adata

results_r = {}

for k in [4, 5, 6, 7, 8, 9, 10, 11, 12, 13]:
    path = f"figures/cme_r_k{k}.tsv"
    if os.path.exists(path):
        df_k = pd.read_csv(path, sep="\t", index_col=0)
        results_r[str(k)] = df_k.iloc[:, 0]
        print(f"K={k}: mean entropy = "
              f"{df_k.iloc[:,0].mean():.4f}")
    else:
        print(f"K={k}: output not found!")

cme_r_df = pd.DataFrame(results_r)
cme_r_df = cme_r_df[[
    str(k) for k in
    sorted([int(c) for c in cme_r_df.columns])
]]
cme_r_df.to_csv("figures/05_cme_r_combined.tsv", sep="\t")

for col in cme_r_df.columns:
    adata.obs[f"CME_R_{col}clusters"] = cme_r_df[col]

# run original Python script

result_k4_py = subprocess.run([
    "python", "cme_script_original.py",
    "-i", "cme_input_all.tsv",
    "-o", "figures/cme_py_k4.tsv",
    "--n_bcs", "2",
    "-n", "4"
], capture_output=True, text=True)

print(f"Python K=4: return code={result_k4_py.returncode}")
if result_k4_py.returncode != 0:
    print(f"ERROR: {result_k4_py.stderr[:200]}")

for k in [5, 6, 7, 8, 9, 10, 11, 12, 13]:
    result = subprocess.run([
        "python", "cme_script_original.py",
        "-i", "cme_input_all.tsv",
        "-o", f"figures/cme_py_k{k}.tsv",
        "--BC_ranking", "BC_rankings.tsv",
        "--n_bcs", "8",
        "-n", str(k)
    ], capture_output=True, text=True)

    print(f"Python K={k}: return code={result.returncode}")
    if result.returncode != 0:
        print(f"ERROR: {result.stderr[:200]}")

# load Python results and add to adata

results_py = {}

for k in [4, 5, 6, 7, 8, 9, 10, 11, 12, 13]:
    path = f"figures/cme_py_k{k}.tsv"
    if os.path.exists(path):
        df_k = pd.read_csv(path, sep="\t", index_col=0)
        results_py[str(k)] = df_k.iloc[:, 0]
        print(f"K={k}: mean entropy = "
              f"{df_k.iloc[:,0].mean():.4f}")
    else:
        print(f"K={k}: output not found!")

cme_py_df = pd.DataFrame(results_py)
cme_py_df = cme_py_df[[
    str(k) for k in
    sorted([int(c) for c in cme_py_df.columns])
]]
cme_py_df.to_csv("figures/05_cme_py_combined.tsv", sep="\t")

for col in cme_py_df.columns:
    adata.obs[f"CME_PY_{col}clusters"] = cme_py_df[col]

# plot R CME maps

r_cols = sorted(
    [c for c in adata.obs.columns
     if c.startswith("CME_R_") and "clusters" in c],
    key=lambda x: int(x.split("_")[2].replace("clusters",""))
)

n_k = len(r_cols)
fig, axes = plt.subplots(1, n_k, figsize=(n_k * 5, 5))

for i, col in enumerate(r_cols):
    n = col.split("_")[2].replace("clusters","")
    sc.pl.embedding(
        adata, basis="spatial", color=col,
        title=f"K={n}", ax=axes[i], show=False,
        color_map="RdYlGn_r", size=10,
        vmin=0.0, vmax=1.8
    )

plt.suptitle("CME Maps — Original R Script", fontsize=14)
plt.tight_layout()
plt.savefig("figures/05_cme_maps_r.png",
            dpi=150, bbox_inches="tight", facecolor="white")
plt.close()
print("Saved: figures/05_cme_maps_r.png")

# plot Python CME maps

py_cols = sorted(
    [c for c in adata.obs.columns
     if c.startswith("CME_PY_") and "clusters" in c],
    key=lambda x: int(x.split("_")[2].replace("clusters",""))
)

fig, axes = plt.subplots(1, n_k, figsize=(n_k * 5, 5))

for i, col in enumerate(py_cols):
    n = col.split("_")[2].replace("clusters","")
    sc.pl.embedding(
        adata, basis="spatial", color=col,
        title=f"K={n}", ax=axes[i], show=False,
        color_map="RdYlGn_r", size=10,
        vmin=0.0, vmax=1.8
    )

plt.suptitle("CME Maps — Original Python Script", fontsize=14)
plt.tight_layout()
plt.savefig("figures/05_cme_maps_py.png",
            dpi=150, bbox_inches="tight", facecolor="white")
plt.close()
print("Saved: figures/05_cme_maps_py.png")

### 5.5 Results

![CME Maps — Original R Script](figures/05_cme_maps_r.png)

![CME Maps — Python Script](figures/05_cme_maps_py.png)


The CME maps were produced for both the R and Python scripts for all cluster counts from K=4 to K=13.

Comparing the two implementations, the R and Python scripts produced noticeably different CME maps. The original R script showed higher entropy (more red) across many regions, while the Python script showed lower entropy (more green). 

### 5.6 Interpretation 

The expectation was that both scripts would produce identical results since they implement the same CME algorithm on the same input data. The fact that they differ suggests that there is an implementation error in one of the scripts.

The R script shows higher entropy overall, which means there is more disagreement between runs than the Python script. This could be caused by an error in the cluster label alignment step, if alignment fails, runs that actually agree would appear to disagree, producing high entropy.

## 6. CME Control Experiments 

### 6.1 Objective 

To investigate the behavior of CME under controlled conditions and verify that it correctly identifies regions of agreement and disagreement between clustering runs. 

Two control experiments were performed:

1. **K=4 vs. K=6** : comparing two runs with different cluster counts.
2. **Label swapping** : comparing two identical K=8 runs where the labels of two clusters were swapped, to test whether CME correctly identifies that the runs are the same.

## 6.2 Control Experiment 1 (K=4 vs. K=6)

### Objective 

To see what happns when CME compares two runs with different cluster counts.

### Method 

Two runs with K=4 and K=6 were selected from the 625 runs. 
Both runs were saved as input and passed to both the Python and modified R CME scripts. The results were then plotted next to eachother. 

### Expectation 

Since the same two runs were used in creating the CME maps using both scripts , we expect to have identical results from the R and Python scripts, where the maps show low entropy (green) in regions of agreement and high entropy (red) in regions of disagreement.

### Code 

In [ ]:
import numpy as np
import pandas as pd
import subprocess
import os
import matplotlib.pyplot as plt
import scanpy as sc

# Select one K=4 and one K=6 run

k4_runs = [c for c in adata.obs.columns
           if c.startswith("sl_seed")
           and adata.obs[c].nunique() == 4]

k6_runs = [c for c in adata.obs.columns
           if c.startswith("sl_seed")
           and adata.obs[c].nunique() == 6]

k4_run = k4_runs[0]
k6_run = k6_runs[0]

print(f"K=4 run: {k4_run}")
print(f"K=6 run: {k6_run}")

# Save one input file for both scripts

combined_df = pd.concat([
    adata.obs[[k4_run]],
    adata.obs[[k6_run]]
], axis=1)
combined_df.to_csv("figures/cme_control_k4_k6.tsv",
                    sep="\t")
print("Saved: figures/cme_control_k4_k6.tsv")
print(f"Columns: {combined_df.columns.tolist()}")

# R and Python calls

r_call = [
    "Rscript", "cme_script_modified.R",
    "-i", "figures/cme_control_k4_k6.tsv",  # ← same file
    "-o", "figures/cme_control_r.tsv",
    "--n_bcs", "2"
]

py_call = [
    "python", "cme_script_original.py",
    "-i", "figures/cme_control_k4_k6.tsv",  # ← same file
    "-o", "figures/cme_control_py.tsv",
    "--n_bcs", "2",
    "--no_split"
]

# Run both scripts

result_r = subprocess.run(r_call, capture_output=True, text=True)
print("\nR return code:", result_r.returncode)
print("R STDERR:", result_r.stderr)

result_py = subprocess.run(py_call, capture_output=True, text=True)
print("\nPython return code:", result_py.returncode)
print("Python STDERR:", result_py.stderr)

# Load results

cme_r  = pd.read_csv("figures/cme_control_r.tsv",
                      sep="\t", index_col=0)
cme_py = pd.read_csv("figures/cme_control_py.tsv",
                      sep="\t", index_col=0)

adata.obs["CME_R_control_k4k6"]  = cme_r.iloc[:, 0]
adata.obs["CME_PY_control_k4k6"] = cme_py.iloc[:, 0]

print("R CME stats:")
print(cme_r.describe())
print("\nPython CME stats:")
print(cme_py.describe())

# Plot results

fig, axes = plt.subplots(1, 4, figsize=(28, 7))

sc.pl.embedding(
    adata, basis="spatial", color=k4_run,
    title=f"K=4 clustering\n{k4_run[8:40]}",
    ax=axes[0], show=False, size=10, legend_loc=None
)
sc.pl.embedding(
    adata, basis="spatial", color=k6_run,
    title=f"K=6 clustering\n{k6_run[8:40]}",
    ax=axes[1], show=False, size=10, legend_loc=None
)
sc.pl.embedding(
    adata, basis="spatial",
    color="CME_R_control_k4k6",
    title="CME — Modified R Script",
    ax=axes[2], show=False,
    color_map="RdYlGn_r", size=10,
    vmin=0.0, vmax=0.693
)
sc.pl.embedding(
    adata, basis="spatial",
    color="CME_PY_control_k4k6",
    title="CME — Python Script",
    ax=axes[3], show=False,
    color_map="RdYlGn_r", size=10,
    vmin=0.0, vmax=0.693
)

plt.suptitle(
    f"Control Experiment 1 — K=4 vs K=6\n"
    f"Same input for both scripts: cme_control_k4_k6.tsv\n"
    f"K=4: {k4_run}\nK=6: {k6_run}",
    fontsize=11
)
plt.tight_layout()
plt.savefig("figures/06_cme_control_k4_k6.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.close()
print("Saved: figures/06_cme_control_k4_k6.png")

### Results 

![Control Experiment 1 — K=4 vs K=6](figures/06_cme_control_k4_k6.png)

As expected, the results from both scripts are similar. Furthermore, we observed correct entropy assignments in regions of agreement / disagreement between the runs.

However, in some regions (e.g. the vertical cluster in the center, V3), the CME maps show intermediate entropy, although low entropy was expected, since the cells from both runs are assigned to the same cluster (with different cluster labels).ls).

### Interpretation

The reason for intermediate entropy in some regions in the control experiment between two runs with K=4 and K=6 is that the number of clusters of these runs are different, therefore some cluster labels exist in K=6 but not in K=4. From the K=4 reference, these labels are invisible and appear to agree, while from the K=6 reference they disagree. Averaging these gives an intermediate value, which is biologically not meaningful. This confirms that CME shouls only be used o compare runs with the same cluster count.

## 6.3 Control Experiment 2 (Label swapping on K=8)

### Objective

To verify that the CME algorithm correctly identifies that two identical runs are the same, even when their cluster labels have been swapped.

### Method 

One K=8 clustering run was selected and a copy was made with two cluster labels swapped (clusters 0 and 3). Both the original and the swapped copy runs were passed to both the R and the Python CME scripts. 

### Expectation 

Since the two runs are identical with only two cluster labels swapped, correct alignment should remap all labels to match, and the entropy should be 0.0 (green) for every cell. 

### Code

In [ ]:
# Pick one K=8 run

k8_cols = [c for c in adata.obs.columns
           if c.startswith("sl_seed")
           and adata.obs[c].nunique() == 8]

k8_run = k8_cols[0]
print(f"K=8 run: {k8_run}")
print(f"Clusters: {sorted(adata.obs[k8_run].astype(str).unique())}")

# Create swapped copy

k8_original = adata.obs[k8_run].astype(str).copy()
k8_swapped  = k8_original.copy()

k8_swapped[k8_swapped == "0"] = "temp"
k8_swapped[k8_swapped == "3"] = "0"
k8_swapped[k8_swapped == "temp"] = "3"

print(f"\nOriginal clusters: {sorted(k8_original.unique())}")
print(f"Swapped clusters:  {sorted(k8_swapped.unique())}")

adata.obs["k8_swapped"] = k8_swapped

# save as input files

orig_df    = adata.obs[[k8_run]].copy()
swapped_df = adata.obs[["k8_swapped"]].copy()

orig_df.to_csv("figures/k8_original.tsv", sep="\t")
swapped_df.to_csv("figures/k8_swapped.tsv", sep="\t")

combined_swap = pd.concat([orig_df, swapped_df], axis=1)
combined_swap.to_csv("figures/cme_swap_input.tsv",
                      sep="\t")
print("Saved input files")

# Define R and Python calls

r_call = [
    "Rscript", "cme_script_original.R",
    "-i", "figures/cme_swap_input.tsv",
    "-o", "figures/cme_swap_r.tsv",
    "--n_bcs", "2",
    "-n", "8"
]

py_call = [
    "python", "cme_script_original.py",
    "-i", "figures/k8_original.tsv",
    "-i", "figures/k8_swapped.tsv",
    "-o", "figures/cme_swap_py.tsv",
    "--n_bcs", "2"
]

# Run both scripts

result_r = subprocess.run(r_call, capture_output=True, text=True)
print("R return code:", result_r.returncode)
print("R STDERR:", result_r.stderr)

result_py = subprocess.run(py_call, capture_output=True, text=True)
print("Python return code:", result_py.returncode)
print("Python STDERR:", result_py.stderr)

# Load results

cme_r_swap  = pd.read_csv("figures/cme_swap_r.tsv",
                            sep="\t", index_col=0)
cme_py_swap = pd.read_csv("figures/cme_swap_py.tsv",
                           sep="\t", index_col=0)

adata.obs["CME_R_swap"]  = cme_r_swap.iloc[:, 0]
adata.obs["CME_PY_swap"] = cme_py_swap.iloc[:, 0]

print(f"R entropy   — mean: {cme_r_swap.iloc[:,0].mean():.4f} "
      f"(expected: 0.0)")
print(f"Python entropy — mean: {cme_py_swap.iloc[:,0].mean():.4f} "
      f"(expected: 0.0)")

# Plot

fig, axes = plt.subplots(1, 4, figsize=(28, 7))

sc.pl.embedding(
    adata, basis="spatial", color=k8_run,
    title=f"K=8 original\n{k8_run[8:40]}",
    ax=axes[0], show=False, size=10, legend_loc=None
)
sc.pl.embedding(
    adata, basis="spatial", color="k8_swapped",
    title="K=8 swapped\n(clusters 0 and 3 swapped)",
    ax=axes[1], show=False, size=10, legend_loc=None
)
sc.pl.embedding(
    adata, basis="spatial", color="CME_R_swap",
    title="CME — R Script\n",
    ax=axes[2], show=False,
    color_map="RdYlGn_r", size=10,
    vmin=0.0, vmax=0.693
)
sc.pl.embedding(
    adata, basis="spatial", color="CME_PY_swap",
    title="CME — Python Script\n",
    ax=axes[3], show=False,
    color_map="RdYlGn_r", size=10,
    vmin=0.0, vmax=0.693
)

plt.suptitle(
    f"Control Experiment 2 — Label Swapping on K=8\n"
    f"Original: {k8_run}\n"
    f"Swapped: clusters 0 ↔ 3",
    fontsize=11
)
plt.tight_layout()
plt.savefig("figures/cme_control_swap.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.close()
print("Saved: figures/cme_control_swap.png")

### Results 

![Control Experiment 2 — Label Swapping](figures/cme_control_swap.png)

As expected the Python script produced entropy 0.0 for all cells and the CME map was entirely green. 
The original R script however produced higher entropy in some regions, meaning the R script incorrectly identifies disagreement between two identical clustering runs. 

### Interpretation 

The label swapping experiment shows the original R script has a bug in the alignment step, which motivates a detailed invesitgation of the alignment function in the R script. 

## 7. CME R Script Debugging 

### 7.1 Objective

To idenfity and fix the bugs in the original R script that caused it to produce different results from the Python implementation, as discovered in the control experiments.

### 7.2 Bug 1 

**Code Block :**

```r
for(j in 1:ncol(bcs)){
    bcs[,j] <- factor(bcs[,j], 
        levels = as.factor(as.numeric(unique(bcs[,j]))))
}
```

**What happens :**

`unique()` returns values in the order of first appearance in the data and not sorted.

Factor levels determine the row and column order of the contingeny table that is passed to the Hungarian algorithm. If the levels are in different order in different runs, the Hungarian algorithm solves a different assignment problem, producing wrong cluster mappings.

**The fix :**

```r
for(j in 1:ncol(bcs)){
    bcs[,j] <- factor(bcs[,j], 
        levels = sort(as.numeric(unique(bcs[,j]))))
}
```

Adding `sort()` orders the cluster labels numerically regardless of which cell appears firt in the data.

### 7.3 Bug 2 

**Code Block :**

```r
hung <- clue::solve_LSAP(table(bcs[,ref], bcs[,i]),
                         maximum = TRUE)
lookup <- cbind(seq_along(hung), levs[[i]][hung])
levels(bcs[,i]) <- lookup[order(as.integer(lookup[,2])),1]
bcs[,i] <- as.factor(as.character(bcs[,i]))
```

**What happens :**

`seq_along(hung)` generates position numbers, which are 1-based. However, the cluster labels from SpatialLeiden are 0-based. The lookup table mixes these position numbers with cluster labels, causing cells to be assigned position numbers as their new cluster labels instead of the correct reference labels. 

**The fix :**

```r
mapping <- levs[[ref]][hung]
names(mapping) <- levs[[i]]
bcs[,i] <- mapping[as.character(bcs[,i])]
bcs[,i] <- factor(bcs[,i], levels=levs[[ref]])
```

This direclty creates a label mapping using the reference cluster labels. 

### 7.4 Bug3

**Code Block :**

```r
hung <- clue::solve_LSAP(table(bcs[,ref], bcs[,i]),
                         maximum = TRUE)
```

**What happens**

`solve_LSAP` assigns each row to a column. In the original script the contingency table is passed with rows = reference clusters and columns = other clusters, meaning `solve_LSAP` assigns each reference cluster to an other cluster
This produces a different result from the Python script.

**The fix :**
```r
ct <- table(bcs[,ref], bcs[,i])
hung <- clue::solve_LSAP(t(ct), maximum = TRUE)
```

Transposing the contingency table before passing it to `solve_LSAP` matches the behavior of Python's implementation.

### 7.5 Verification - Control Experiment Label Swapping

The bugs were verified by repeating the label swapping control experiment from section 6 using 3 scripts: the original R script, the fixed R script, and the Python script. 
The same K=8 run was used with clusters 0 and 3 swaped. Since the two runs are identical, an entropy of 0.0 (all green in the CME maps) is expected. 

In [ ]:
import numpy as np
import pandas as pd
import subprocess
import matplotlib.pyplot as plt
import scanpy as sc
import os

sc.settings.autoshow = False

# Use same K=8 run from section 6

k8_cols = [c for c in adata.obs.columns
           if c.startswith("sl_seed")
           and adata.obs[c].nunique() == 8]

k8_run = k8_cols[0]

# Create swapped copy 

k8_original = adata.obs[k8_run].astype(str).copy()
k8_swapped  = k8_original.copy()

k8_swapped[k8_swapped == "0"] = "temp"
k8_swapped[k8_swapped == "3"] = "0"
k8_swapped[k8_swapped == "temp"] = "3"

adata.obs["k8_swapped"] = k8_swapped

# Save as input files 

orig_df    = adata.obs[[k8_run]].copy()
swapped_df = adata.obs[["k8_swapped"]].copy()

orig_df.to_csv("figures/k8_original.tsv",
               sep="\t")
swapped_df.to_csv("figures/k8_swapped.tsv",
                  sep="\t")

combined_swap = pd.concat([orig_df, swapped_df], axis=1)
combined_swap.to_csv("figures/cme_swap_input.tsv",
                      sep="\t")
print("Saved input files")

# Define call for all 3 scripts 

# original R script
r_orig_call = [
    "Rscript", "cme_script_original.R",
    "-i", "figures/cme_swap_input.tsv",
    "-o", "figures/cme_swap_r_orig.tsv",
    "--n_bcs", "2",
    "-n", "8"
]

# fixed R script (3 bugs fixed)
r_fixed_call = [
    "Rscript", "cme_script_fix3bugs.R",
    "-i", "figures/cme_swap_input.tsv",
    "-o", "figures/cme_swap_r_fixed.tsv",
    "--n_bcs", "2",
    "-n", "8"
]

# Python script
py_call = [
    "python", "cme_script_original.py",
    "-i", "figures/k8_original.tsv",
    "-i", "figures/k8_swapped.tsv",
    "-o", "figures/cme_swap_py.tsv",
    "--n_bcs", "2"
]

# Run all scripts

result_r_orig = subprocess.run(r_orig_call,
                                capture_output=True,
                                text=True)
print("\nOriginal R return code:", result_r_orig.returncode)
print("Original R STDERR:", result_r_orig.stderr)

result_r_fixed = subprocess.run(r_fixed_call,
                                 capture_output=True,
                                 text=True)
print("\nFixed R return code:", result_r_fixed.returncode)
print("Fixed R STDERR:", result_r_fixed.stderr)

result_py = subprocess.run(py_call,
                            capture_output=True,
                            text=True)
print("\nPython return code:", result_py.returncode)
print("Python STDERR:", result_py.stderr)

# Load results 

cme_r_orig  = pd.read_csv(
    "figures/cme_swap_r_orig.tsv",
    sep="\t", index_col=0
)
cme_r_fixed = pd.read_csv(
    "figures/cme_swap_r_fixed.tsv",
    sep="\t", index_col=0
)
cme_py = pd.read_csv(
    "figures/cme_swap_py.tsv",
    sep="\t", index_col=0
)

adata.obs["CME_SWAP_R_ORIG"]  = cme_r_orig.iloc[:, 0]
adata.obs["CME_SWAP_R_FIXED"] = cme_r_fixed.iloc[:, 0]
adata.obs["CME_SWAP_PY"]      = cme_py.iloc[:, 0]

# Plot all maps

fig, axes = plt.subplots(1, 5, figsize=(35, 7))

# original clustering
sc.pl.embedding(
    adata, basis="spatial", color=k8_run,
    title=f"K=8 original\n{k8_run[8:40]}",
    ax=axes[0], show=False, size=10,
    legend_loc=None
)

# swapped clustering
sc.pl.embedding(
    adata, basis="spatial", color="k8_swapped",
    title="K=8 swapped\n",
    ax=axes[1], show=False, size=10,
    legend_loc=None
)

# original R CME
sc.pl.embedding(
    adata, basis="spatial",
    color="CME_SWAP_R_ORIG",
    title="CME — Original R\n)",
    ax=axes[2], show=False,
    color_map="RdYlGn_r", size=10,
    vmin=0.0, vmax=0.693
)

# fixed R CME
sc.pl.embedding(
    adata, basis="spatial",
    color="CME_SWAP_R_FIXED",
    title="CME — Fixed R\n)",
    ax=axes[3], show=False,
    color_map="RdYlGn_r", size=10,
    vmin=0.0, vmax=0.693
)

# Python CME
sc.pl.embedding(
    adata, basis="spatial",
    color="CME_SWAP_PY",
    title="CME — Python\n",
    ax=axes[4], show=False,
    color_map="RdYlGn_r", size=10,
    vmin=0.0, vmax=0.693
)

plt.suptitle(
    f"Label Swapping Verification — K=8\n"
    f"Original: {k8_run}\n"
    f"Swapped: clusters 0 and 3\n"
    f"Expected entropy: 0.0 for all cells",
    fontsize=11
)
plt.tight_layout()
plt.savefig("figures/cme_swap_verification.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.close()
print("Saved: figures/cme_swap_verification.png")

**Result :**

![Label Swap Verification](figures/cme_swap_verification.png)

The CME map produced from the fixed R script correctly shows entropy of 0.0 (green) in all cells, which confirms that the alignment correctly remapped the swapped labels , the same result as the Python script 

### 7.6 Verification - Full CME Maps

After fixing all 3 bugs the fixed R scripts was run on all 625 clustering runs and compared to the Python output using correlation per count count.

In [ ]:
import numpy as np
import pandas as pd
import subprocess
import matplotlib.pyplot as plt
import scanpy as sc
import os

# Run fixed R script per K

# K=4 separately — only 2 runs available
result_k4 = subprocess.run([
    "Rscript", "cme_script_fix3bugs.R",
    "-i", "cme_input_all.tsv",
    "-o", "figures/cme_fix3_k4.tsv",
    "--n_bcs", "2",
    "-n", "4"
], capture_output=True, text=True)

print(f"Fixed R K=4: return code={result_k4.returncode}")
if result_k4.returncode != 0:
    print(f"ERROR: {result_k4.stderr[:200]}")

# K=5 to K=13 with BC_rankings
for k in [5, 6, 7, 8, 9, 10, 11, 12, 13]:
    result = subprocess.run([
        "Rscript", "cme_script_fix3bugs.R",
        "-i", "cme_input_all.tsv",
        "-o", f"figures/cme_fix3_k{k}.tsv",
        "--BC_ranking", "BC_rankings.tsv",
        "--n_bcs", "8",
        "-n", str(k)
    ], capture_output=True, text=True)

    print(f"Fixed R K={k}: return code={result.returncode}")
    if result.returncode != 0:
        print(f"ERROR: {result.stderr[:200]}")

# Load and add to adata

results_fix3 = {}

for k in [4, 5, 6, 7, 8, 9, 10, 11, 12, 13]:
    path = f"figures/cme_fix3_k{k}.tsv"
    if os.path.exists(path):
        df_k = pd.read_csv(path, sep="\t", index_col=0)
        results_fix3[str(k)] = df_k.iloc[:, 0]
        print(f"K={k}: mean entropy = "
              f"{df_k.iloc[:,0].mean():.4f}")
    else:
        print(f"K={k}: output not found!")

r_fixed_df = pd.DataFrame(results_fix3)
r_fixed_df = r_fixed_df[[
    str(k) for k in
    sorted([int(c) for c in r_fixed_df.columns])
]]

for col in r_fixed_df.columns:
    adata.obs[f"CME_FIX3_{col}clusters"] = r_fixed_df[col]
    print(f"Added: CME_FIX3_{col}clusters")

# Plot fixed R CME maps

fix3_cols = sorted(
    [c for c in adata.obs.columns
     if c.startswith("CME_FIX3_")
     and "clusters" in c],
    key=lambda x: int(x.split("_")[2].replace("clusters",""))
)

n_k = len(fix3_cols)
fig, axes = plt.subplots(1, n_k, figsize=(n_k * 5, 5))

for i, col in enumerate(fix3_cols):
    n = col.split("_")[2].replace("clusters","")
    sc.pl.embedding(
        adata, basis="spatial", color=col,
        title=f"K={n}", ax=axes[i], show=False,
        color_map="RdYlGn_r", size=10,
        vmin=0.0, vmax=1.8
    )

plt.suptitle("CME Maps — Fixed R Script (3 bugs fixed)",
             fontsize=14)
plt.tight_layout()
plt.savefig("figures/07_cme_maps_fix3bugs.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.close()
print("Saved: figures/07_cme_maps_fix3bugs.png")

# Plot original R vs fixed R vs Python

orig_r_cols = sorted(
    [c for c in adata.obs.columns
     if c.startswith("CME_R_")
     and "clusters" in c
     and "control" not in c],
    key=lambda x: int(x.split("_")[2].replace("clusters",""))
)

py_cols = sorted(
    [c for c in adata.obs.columns
     if c.startswith("CME_PY_")
     and "clusters" in c
     and "control" not in c],
    key=lambda x: int(x.split("_")[2].replace("clusters",""))
)

common_ks = sorted(
    set([c.split("_")[2].replace("clusters","")
         for c in orig_r_cols]) &
    set([c.split("_")[2].replace("clusters","")
         for c in fix3_cols]) &
    set([c.split("_")[2].replace("clusters","")
         for c in py_cols]),
    key=lambda x: int(x)
)

n_k = len(common_ks)
fig, axes = plt.subplots(3, n_k,
                          figsize=(n_k * 5, 15))

for i, k in enumerate(common_ks):
    sc.pl.embedding(
        adata, basis="spatial",
        color=f"CME_R_{k}clusters",
        title=f"K={k}", ax=axes[0, i], show=False,
        color_map="RdYlGn_r", size=10,
        vmin=0.0, vmax=1.8
    )
    sc.pl.embedding(
        adata, basis="spatial",
        color=f"CME_FIX3_{k}clusters",
        title=f"K={k}", ax=axes[1, i], show=False,
        color_map="RdYlGn_r", size=10,
        vmin=0.0, vmax=1.8
    )
    sc.pl.embedding(
        adata, basis="spatial",
        color=f"CME_PY_{k}clusters",
        title=f"K={k}", ax=axes[2, i], show=False,
        color_map="RdYlGn_r", size=10,
        vmin=0.0, vmax=1.8
    )

axes[0, 0].set_ylabel("Original R",
                       fontsize=12, fontweight="bold")
axes[1, 0].set_ylabel("Fixed R\n(3 bugs fixed)",
                       fontsize=12, fontweight="bold")
axes[2, 0].set_ylabel("Python",
                       fontsize=12, fontweight="bold")

plt.suptitle(
    "CME Maps — Original R vs Fixed R vs Python",
    fontsize=14, fontweight="bold"
)
plt.tight_layout()
plt.savefig("figures/07_cme_maps_comparison.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.close()
print("Saved: figures/07_cme_maps_comparison.png")

![Fixed R CME Maps](figures/07_cme_maps_fix3bugs.png)

![Comparison All Three](figures/07_cme_maps_comparison.png)

**Result :**

As expected , the CME maps are the same as the CME maps from the Python 
script , indicating that the alignment function now works correctly.

## 8. Biological Interpretation

### 8.1 Objective

To evaluate whether the CME maps produced by the corrected R implementation correspond to known biological structures in the mouse hypothalamus preoptic region, and to assess which cluster count produces the most biologically meaningful result.

### 8.2 Background 

A biologically meaningful CME map should reflect 
the known organization of the tissue. In the mouse 
hypothalamus preoptic region, well-defined 
anatomical structures such as the third ventricle 
(V3) and the bed nucleus of the stria terminalis 
(BST) should appear as stable regions since they 
are consistently identifiable regardless of 
parameter choice. In contrast, boundaries between 
structures should 
appear as unstable regions.

If the CME maps show this pattern, green in known 
structures and red at boundaries, it confirms that 
the corrected implementation is working correctly 
and that the clustering results are biologically 
meaningful.


### 8.3 Method

The CME maps produced by the corrected R script 
for all cluster counts from K=4 to K=13 were 
compared visually to the known anatomical atlas 
of the mouse hypothalamus preoptic region. 
Stable regions (green) and unstable regions (red) 
in the CME maps were identified and compared to 
the known locations of anatomical structures in 
the atlas.

### 8.4 Expectation 

We expect known brain regions to show low entropy at appropriate K values, where the resolution is fine enough to seperate them from adjacent structures and the boundaries between known structures to show higher entropy, since cells in these transition zones are harder to assign consistently across parameter combinations. 

### 8.5 Results

![CME Maps Atlas](figures/08_cme_maps_atlas.png)

### 8.6 Interpretation 

As expected, known brain region show low entropy, while the boundaries between these regions show higher entropy across most K values, since cells in these transition zones share gene expression properties of adjacent regions and are assigned inconsistently across parameter combinations.
At lower K values (K=4 to K=6), many boundary regions appear red because the resolution is too coarse to seperate adjacent structures cleanly.

K=8 to K=9 appears to provide the best balance between resolution and stability, these K values consistently identify the major known brain structures while keeping entropy low in well-defined regions. At higher K values the overall entropy increases. 

The CME maps produced by the corrected 
implementation show a biologically meaningful 
pattern that corresponds to the known anatomical 
organization of the mouse hypothalamus preoptic 
region. Stable regions match known brain structures 
and unstable regions correspond to boundaries 
between them. This validates both the corrected 
CME implementation and the SpatialLeiden clustering 
approach used in this project.

## 9. Summary and Conclusions 

### 9.1 What was done 

In this project, the SpatialLeiden clustering 
algorithm was applied to the MERFISH mouse 
hypothalamus preoptic region dataset. A parameter 
scan was performed by running 625 clustering 
combinations across four parameters (layer ratio, 
latent resolution, spatial resolution and random 
seed) to explore the effect of parameter choice 
on the clustering results.

The Cross Method Entropy (CME) pipeline was then 
applied to quantify the stability of clustering 
results across all parameter combinations, 
producing stability maps for each cluster count 
from K=4 to K=13.

### 9.2 Key findings 


**1. Spatial clustering is highly parameter-sensitive**

The parameter scan showed that all three 
SpatialLeiden parameters significantly affect 
the clustering results, confirming that spatial 
clustering is an exploratory task with no single 
correct answer. Higher resolution values produce 
more clusters and higher layer ratios produce 
more spatially coherent clusters.

**2. Three bugs were found and fixed in the 
original CME R script**

During the CME analysis, a discrepancy was 
discovered between the R and Python 
implementations. Three bugs were identified in 
the align_classes function of the original R 
script:

- Bug 1: wrong factor level ordering (unique 
  instead of sort)
- Bug 2: wrong mapping application (position 
  numbers used as cluster labels)
- Bug 3: wrong solve_LSAP orientation 

The bugs were verified using a label swapping 
control experiment (two identical runs with 
swapped labels must produce entropy = 0.0). The 
original R script produced entropy > 0, while the 
fixed R script and Python both produced 0.0.

**3. The corrected CME maps are biologically 
meaningful**

After fixing all three bugs, the CME maps show 
a pattern that corresponds to the known 
anatomical organization of the mouse hypothalamus 
preoptic region. Stable regions (green) match 
known brain structures such as V3 and BST, while 
unstable regions (red) correspond to boundaries 
between structures. K=8 to K=9 appears to provide 
the most biologically meaningful clustering for 
this dataset.